In [1]:
import sys; sys.path.append("..")
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
from src.data import load_data

X_train, X_test, y_train, y_test, labels = load_data()

sc = StandardScaler().fit(X_train)
Ztr = sc.transform(X_train)
Zte = sc.transform(X_test)

nn = NearestNeighbors(n_neighbors=1, n_jobs=-1).fit(Ztr)
d, idx = nn.kneighbors(Zte)
d = d[:, 0]
nhan_lang_gieng = y_train.to_numpy()[idx[:, 0]]
cung_lop = nhan_lang_gieng == y_test.to_numpy()

ten = y_test.map(dict(enumerate(labels))).to_numpy()
bang = pd.DataFrame({"lop": ten, "khoang_cach": d, "cung_lop": cung_lop})
tom_tat = bang.groupby("lop").agg(
    so_mau=("khoang_cach", "size"),
    kc_trung_vi=("khoang_cach", "median"),
    kc_p90=("khoang_cach", lambda s: s.quantile(0.9)),
    ti_le_hang_xom_cung_lop=("cung_lop", "mean"),
).round(4)
tom_tat.to_csv("../results/test/nearest_neighbor_gap.csv")
print("Do chinh xac 1-NN tren test:", round(cung_lop.mean(), 4))
tom_tat

Do chinh xac 1-NN tren test: 0.9972


,so_mau,kc_trung_vi,kc_p90,ti_le_hang_xom_cung_lop
lop,,,,
ARP_poisioning,1525,0.0410,2.4581,0.9803
DDOS_Slowloris,107,0.0022,0.0060,1.0000
DOS_SYN_Hping,18018,0.0001,0.0006,1.0000
MQTT_Publish,828,0.0426,0.1687,0.9988
Metasploit_Brute_Force_SSH,7,0.1381,0.3425,1.0000
NMAP_FIN_SCAN,6,0.0001,0.1088,1.0000
NMAP_OS_DETECTION,400,0.0100,0.0874,1.0000
NMAP_TCP_scan,200,0.0936,0.3306,1.0000
NMAP_UDP_SCAN,517,0.0002,0.0136,0.9903


In [2]:
hiem = ["Metasploit_Brute_Force_SSH", "NMAP_FIN_SCAN", "Wipro_bulb"]
m = bang["lop"].isin(hiem).to_numpy()
chi_tiet = bang[m].assign(lop_lang_gieng=[labels[i] for i in nhan_lang_gieng[m]])
chi_tiet.sort_values(["lop", "khoang_cach"]).round(4)

,lop,khoang_cach,cung_lop,lop_lang_gieng
17619,Metasploit_Brute_Force_SSH,0.0233,True,Metasploit_Brute_Force_SSH
21383,Metasploit_Brute_Force_SSH,0.0724,True,Metasploit_Brute_Force_SSH
5822,Metasploit_Brute_Force_SSH,0.0973,True,Metasploit_Brute_Force_SSH
22054,Metasploit_Brute_Force_SSH,0.1381,True,Metasploit_Brute_Force_SSH
14600,Metasploit_Brute_Force_SSH,0.3220,True,Metasploit_Brute_Force_SSH
14813,Metasploit_Brute_Force_SSH,0.3300,True,Metasploit_Brute_Force_SSH
14056,Metasploit_Brute_Force_SSH,0.3612,True,Metasploit_Brute_Force_SSH
5660,NMAP_FIN_SCAN,0.0001,True,NMAP_FIN_SCAN
19158,NMAP_FIN_SCAN,0.0001,True,NMAP_FIN_SCAN
22289,NMAP_FIN_SCAN,0.0001,True,NMAP_FIN_SCAN
